# WEARO AI Lab — FASHN VTON 1.5 Colab

This notebook reproduces the same FASHN VTON 1.5 inference path used by WEARO's `gpu-tryon` service. It is an R&D benchmark, not the production API.

**Baseline:** tops · model garment · 30 timesteps · guidance 1.5 · seed 42.

In [ ]:
!nvidia-smi


In [ ]:
%cd /content
!rm -rf fashion-dream-try-on-shop
!git clone --branch feature/product-admin --depth 1 https://github.com/minhquanto82kt/fashion-dream-try-on-shop.git
%cd /content/fashion-dream-try-on-shop/fashion-dream-try-on-main/gpu-tryon
!git rev-parse HEAD


In [ ]:
!pip install -q -r requirements.txt
!pip install -q huggingface_hub


In [ ]:
import os
os.environ['VTON_WEIGHTS_DIR'] = '/content/fashn-vton-models'
os.environ['VTON_OUTPUT_DIR'] = '/content/tryon-output'
!python scripts/download_weights.py


In [ ]:
from google.colab import files
from pathlib import Path
uploaded = files.upload()
image_files = [Path('/content') / name for name in uploaded if Path('/content', name).suffix.lower() in {'.jpg', '.jpeg', '.png', '.webp'}]
assert len(image_files) >= 2, 'Upload at least 2 image files: person image and garment image.'
print('Uploaded image files:', [p.name for p in image_files])
print('Use the first image as PERSON_IMAGE and the second as GARMENT_IMAGE, or edit the next cell.')


In [ ]:
PERSON_IMAGE = str(image_files[0])
GARMENT_IMAGE = str(image_files[1])
CATEGORY = 'tops'
GARMENT_PHOTO_TYPE = 'model'
NUM_TIMESTEPS = 30
GUIDANCE_SCALE = 1.5
SEED = 42
print(PERSON_IMAGE, GARMENT_IMAGE)


In [ ]:
from pathlib import Path
from PIL import Image
from fashn_vton import TryOnPipeline
import time
import torch

person = Image.open(PERSON_IMAGE).convert('RGB')
garment = Image.open(GARMENT_IMAGE).convert('RGB')
print('Person:', person.size)
print('Garment:', garment.size)

if torch.cuda.is_available():
    torch.cuda.reset_peak_memory_stats()
    print('GPU:', torch.cuda.get_device_name(0))

pipeline = TryOnPipeline(weights_dir='/content/fashn-vton-models')
start = time.perf_counter()
result = pipeline(
    person_image=person,
    garment_image=garment,
    category=CATEGORY,
    garment_photo_type=GARMENT_PHOTO_TYPE,
    num_samples=1,
    num_timesteps=NUM_TIMESTEPS,
    guidance_scale=GUIDANCE_SCALE,
    seed=SEED,
    segmentation_free=True,
)
elapsed = time.perf_counter() - start
assert result.images, 'FASHN VTON returned no images'
Path('/content/tryon-output').mkdir(parents=True, exist_ok=True)
output_path = '/content/tryon-output/tryon-result.png'
result.images[0].save(output_path, format='PNG')
print(f'Inference time: {elapsed:.2f}s')
if torch.cuda.is_available():
    print(f'Peak allocated VRAM: {torch.cuda.max_memory_allocated()/1024**3:.2f} GB')
    print(f'Peak reserved VRAM: {torch.cuda.max_memory_reserved()/1024**3:.2f} GB')
print('Output:', output_path)


In [ ]:
from IPython.display import display
display(Image.open(PERSON_IMAGE))
display(Image.open(GARMENT_IMAGE))
display(Image.open('/content/tryon-output/tryon-result.png'))


## Evaluation checklist

Record these after every run:

- **Garment fidelity:** shape, print, color and major details preserved?
- **Body/pose:** original pose and proportions preserved?
- **Face/hair:** unchanged except unavoidable generation artifacts?
- **Hands/edges:** fingers, sleeves, neckline and garment boundaries clean?
- **Background:** stable without unwanted deformation?
- **Runtime:** seconds reported above.
- **Peak VRAM:** GB reported above.

Do not commit personal/customer images or generated images to GitHub.

In [ ]:
# Optional controlled parameter benchmark. Keep the same input pair and seed.
EXPERIMENTS = [
    {'num_timesteps': 30, 'guidance_scale': 1.5},
    {'num_timesteps': 40, 'guidance_scale': 1.5},
    {'num_timesteps': 30, 'guidance_scale': 2.0},
]
print(EXPERIMENTS)
print('Run one configuration at a time and record runtime, VRAM and visual-quality observations.')


## Promotion rule

A successful Colab run is evidence for R&D only. Do not automatically promote parameters or model changes to production. Any validated change must be implemented in `gpu-tryon/` or `python/`, then tested against the existing API and Supabase job/storage contract.

Production flow: `WEARO → Python FastAPI → gpu-tryon → FASHN VTON → Supabase Storage → signed URL`.